## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 1
# To reduce runtime, you will sample 20% of the interaction data from user_artist_data.txt using seed=314 (and use this seed throughout the lab)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Before doing anything, create a Spark session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Define the schema for the file to import.  
# -- I added this after reviewing the data file and running an initial import.
# -- Even though I saw the file had no headers, I didn't initially stop to think I needed to add them!  :)
from pyspark.sql.types import StructType, StructField, IntegerType
autoscrobbler_schema = StructType([
                                    StructField("userid", IntegerType()),
                                    StructField("artistid", IntegerType()),
                                    StructField("playcount", IntegerType()),
                                    ])
     

# Load the data.
# The columns are separated by a space, so need to include sep=" ".
df_user = spark.read.csv("/standard/ds7200-apt4c/large_datasets/user_artist_data.txt",  sep=" ", header=False, schema=autoscrobbler_schema)

# Check to see if any columns have null values.
df_user.createOrReplaceTempView("df_user")
df_user_null = spark.sql("""
    SELECT COUNT(*) AS RowsWithNulls
    FROM df_user
    WHERE 
        userid IS NULL 
        OR artistid IS NULL
        OR playcount IS NULL
""")
print(f"""There are {df_user_null.show()} rows with NULL values.""")

# Select a 20% sample from the data.
df_user_small = df_user.sample(fraction=0.20, seed=314)

# Confirm rows were imported.
print(f"""There are {df_user_small.count()} rows in the imported data set.""")

# Review a few records and the schema to make sure it was applied correctly.
# And it does, so yay!
df_user.show(5)
df_user.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 23:29:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-------------+
|RowsWithNulls|
+-------------+
|            0|
+-------------+

There are None rows with NULL values.


[Stage 3:>                                                          (0 + 8) / 8]

There are 4861443 rows in the imported data set.
+-------+--------+---------+
| userid|artistid|playcount|
+-------+--------+---------+
|1000002|       1|       55|
|1000002| 1000006|       33|
|1000002| 1000007|        8|
|1000002| 1000009|      144|
|1000002| 1000010|      314|
+-------+--------+---------+
only showing top 5 rows
root
 |-- userid: integer (nullable = true)
 |-- artistid: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [3]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 2
# Next, remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Remove users that listen to fewer than 10 artists using SQL.
# -- First register the DF as a temp view first.
df_user_small.createOrReplaceTempView("df_user_small")

# See how may users have listened to fewer than 10 artists.

# Select the columns needed for this analysis.
df_user_go = spark.sql("""
SELECT *
FROM df_user_small
WHERE userid IN
    (
        SELECT userid
        FROM df_user_small
        GROUP BY userid
        HAVING COUNT(DISTINCT artistid) > 9
    )
""")

print(f"""There are {df_user_go.count()} rows after removing users that listened to less than 10 artists.""")

[Stage 14:>                                                         (0 + 8) / 8]

There are 4685471 rows after removing users that listened to less than 10 artists.


In [4]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 3
# Lastly, split the data: 60% for training / 20% for validation / 20% for testing
# ----------------------------------------------------------------------------------------------------------------------------------------------
# Split the dataset.
trainDF, valDF, testDF = df_user_go.randomSplit([.6, .2, .2], seed=314)
print(f"""There are {trainDF.count()} rows in the training set, {valDF.count()} rows in the validation set, and {testDF.count()} in the test set.""")

[Stage 49:==================================================>       (7 + 1) / 8]

There are 2809638 rows in the training set, 938020 rows in the validation set, and 937813 in the test set.


In [5]:
# TASK 1 (1 PT)
# Show the first row from the training set
trainDF.show(1)

# Result should remain constant with seed=314.
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|      40|        1|
#  +------+--------+---------+

[Stage 62:=====================>                                    (3 + 5) / 8]

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   321|      40|        1|
+------+--------+---------+
only showing top 1 row


In [6]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
testDF.show(1)

# Result should remain constant with seed=314.
#  +------+--------+---------+
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|     357|        3|
#  +------+--------+---------+

[Stage 70:=====================>                                    (3 + 5) / 8]

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   321|     357|        3|
+------+--------+---------+
only showing top 1 row


In [7]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# The parameters above do not align to the old MLlib library, so using the newer ML library.
from pyspark.ml.recommendation import ALS

# Create the model specification using the parameters for ML.
als_spec = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    rank=8,
    maxIter=5,
    regParam=0.02,
    alpha=0.01
)

# Fit the model.
als_model = als_spec.fit(trainDF)

print(f"""The model has been trained.""")

26/10/10 23:31:34 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/10/10 23:31:34 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


The model has been trained.


In [8]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1
als_predictions = als_model.transform(testDF)

# Use SQL to get 10 predictions for Teset User 1 (userid = 321).
als_predictions.createOrReplaceTempView("als_predictions")

# This will get the top 10 predictions, and will be used again in Task5.
test_user1_top10 = spark.sql("""
SELECT *
FROM als_predictions
WHERE userid=321
AND NOT ISNAN(prediction)
ORDER BY prediction DESC
LIMIT 10
""")

test_user1_top10.show()

[Stage 289:==========================================>              (6 + 2) / 8]

+------+--------+---------+-----------+
|userid|artistid|playcount| prediction|
+------+--------+---------+-----------+
|   321| 1001779|      337| 0.06961893|
|   321|     435|       51| 0.04556962|
|   321| 1000230|        1|0.040184774|
|   321|     868|        1| 0.03997473|
|   321|     357|        3|0.038499746|
|   321|    5811|        1|0.024704734|
|   321|    1864|       25| 0.02387565|
|   321|    2752|        3|0.014052607|
|   321| 1008602|       28|0.008072598|
|   321| 1007540|        7|0.007133397|
+------+--------+---------+-----------+



**TASK 4 (1 PT)**  
**Briefly explain Recall@K in the context of recommender systems**

Recall@k is like the concept of recall in class imbalance problems which is the number of True Positives (TP) that are correctly predicted, e.g., TP / (TP + FN).  What's different for recommendation problems is that the equivalent of True Positives (the numerator) is the top number (k) of the total predicted items of which the user did interact with and which the model correctly predicted as such, and the denominator is this number plus all the other items that the user interacted with.

In [22]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1

# We want to know "of all the artists that the user listened to, how many of the top-k predictions did the model correctly predict."

# One way to do this is with SQL.
# I'll stick with k=10 for this.

# We need to query the test data to get this calc.
testDF.createOrReplaceTempView("testDF")

# First, get the total rows for userid = 321.
all_items = spark.sql("""
SELECT *
FROM testDF
WHERE userid = 321
""")

# Next, get the total of these that were in the predicted set; these are the true positives (tp).
# This part uses the top k=10 from TASK 3.
all_items.createOrReplaceTempView("all_items")
test_user1_top10.createOrReplaceTempView("test_user1_top10")
tp = spark.sql("""
SELECT *
FROM test_user1_top10 AS t
JOIN all_items AS a
ON t.artistid = a.artistid
""")

# Now, calculate Recall*k
tp.createOrReplaceTempView("tp")
recall_k = spark.sql("""
SELECT 
    ROUND(
            (SELECT COUNT(*) FROM tp) /
            (SELECT COUNT(*) FROM all_items)
            ,4
        ) AS RecallK
""")

In [24]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, alpha) to maximize Recall@K

# Build the hyperparameter tuning grid.
from pyspark.ml.tuning import ParamGridBuilder

# I had MS Copilot help recommend values for these parameters.
paramGrid = ParamGridBuilder() \
    .addGrid(als_model.rank, [10, 20, 40]) \        # Specifies different number of hidden features to look for; these values go for low, med, high rank.
    .addGrid(als_model.maxIter, [10, 20]) \         # Specifies how many times to loop through the data.
    .addGrid(als_model.regParam, [0.01, 0.1]) \     # This is the regularization penalty; 0.01 is flexible and 0.1 is stricter (less-overfitting).
    .addGrid(als_model.alpha, [1.0, 10.0]) \        # This is the how much weight is placed on the playcounts data; 1.0 a little weight, 10.0 a lot of weight.
    .build()

In [6]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam, alpha) along with the Recall@K from the validation set

from pyspark.ml.tuning import CrossValidator

crossval = CrossValidator(
    estimator=als,
    estimatorParamMaps=paramGrid,
    evaluator= ???
    numFolds=2
)

In [27]:
# Retrain the best configuration on train + validation data.


In [7]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.


In [8]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?
